# ICMP Tunneling Detection


This notebook we will be attempting to build an ML model that can detect ICMP tunneling based on the trained data.

In [15]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('FINAL_LABELLED_ICMP.csv')

df.head()

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,10.0.2.15-172.217.26.14-33032-443-6,10.0.2.15,33032,172.217.26.14,443,6,05/05/2026 10:29:30 AM,78653,2,3,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,10.0.2.15-142.250.207.142-34886-443-6,10.0.2.15,34886,142.250.207.142,443,6,05/05/2026 10:29:31 AM,29234,3,4,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,10.0.2.15-142.250.195.14-42292-443-6,10.0.2.15,42292,142.250.195.14,443,6,05/05/2026 10:29:34 AM,75876,3,4,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,10.0.2.15-34.120.208.123-47676-443-6,10.0.2.15,47676,34.120.208.123,443,6,05/05/2026 10:30:02 AM,44007,2,3,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,10.0.2.15-142.251.42.234-37546-443-6,10.0.2.15,37546,142.251.42.234,443,6,05/05/2026 10:30:17 AM,310781,16,18,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


## EDA

In [3]:
df.columns

Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Len

In [4]:
df.describe()

c:\Users\magan\miniforge3\envs\hpe_env\Lib\site-packages\pandas\core\nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\magan\miniforge3\envs\hpe_env\Lib\site-packages\pandas\core\nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Src Port,Dst Port,Protocol,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,...,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,36210.000000,36210.000000,36210.000000,3.621000e+04,36210.000000,36210.000000,3.621000e+04,3.621000e+04,36210.000000,36210.000000,...,36210.000000,36210.000000,3.621000e+04,3.621000e+04,3.621000e+04,3.621000e+04,3.621000e+04,3.621000e+04,3.621000e+04,3.621000e+04
mean,12698.920657,198.464954,4.373019,6.385681e+07,19.514941,15.535018,4.635719e+03,1.897719e+04,136.982712,19.078431,...,2.947307,3.348246,3.379876e+06,2.077467e+06,6.150744e+06,1.581153e+06,8.292385e+06,2.910051e+06,1.308333e+07,5.811723e+06
std,21268.360259,2508.863885,7.101363,5.315838e+07,104.433433,326.306521,2.376370e+05,4.715652e+05,880.836197,71.077536,...,49.950969,5.938351,4.126668e+06,3.019127e+06,7.558786e+06,2.607355e+06,1.181133e+07,5.144885e+06,1.516455e+07,1.097745e+07
min,0.000000,0.000000,0.000000,0.000000e+00,1.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000,0.000000,0.000000,2.728640e+06,3.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000,0.000000,0.000000,9.011214e+07,14.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,3.010922e+06,4.626085e+05,3.925996e+06,1.800082e+06,8.161360e+06,2.022023e+06,1.310533e+07,5.554310e+06
75%,35196.000000,53.000000,6.000000,1.163778e+08,21.000000,1.000000,3.200000e+01,3.900000e+01,31.000000,0.000000,...,0.000000,8.000000,5.757658e+06,3.452628e+06,1.067476e+07,1.974944e+06,1.018538e+07,4.219853e+06,1.900203e+07,5.750960e+06
max,65261.000000,60310.000000,17.000000,1.200000e+08,4494.000000,17822.000000,2.549826e+07,4.905276e+07,65160.000000,5840.000000,...,3836.000000,40.000000,1.078944e+08,5.854058e+07,1.078944e+08,1.078944e+08,1.196240e+08,7.657851e+07,1.196240e+08,1.196240e+08


In [5]:
import numpy as np

missing_values = df.isnull().sum()

inf_values = df.isin([np.inf, -np.inf]).sum()

summary = pd.DataFrame({
    "Missing_Values": missing_values,
    "Inf_Values": inf_values
})

summary = summary[
    (summary["Missing_Values"] > 0) | 
    (summary["Inf_Values"] > 0)
]

print(summary)

                Missing_Values  Inf_Values
Flow Bytes/s                21           2
Flow Packets/s               0          23


In [6]:
problem_rows = df[
    df.isnull().any(axis=1) |
    df.isin([np.inf, -np.inf]).any(axis=1)
]

problem_rows

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
581,134.221.96.0-0.0.1.51-0-0-0,134.221.96.0,0,0.0.1.51,0,0,05/05/2026 10:49:46 AM,0,2,0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
806,49.44.220.94-10.0.2.15-443-48568-6,49.44.220.94,443,10.0.2.15,48568,6,05/05/2026 11:02:10 AM,0,2,0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
809,49.44.220.82-10.0.2.15-443-46714-6,49.44.220.82,443,10.0.2.15,46714,6,05/05/2026 11:02:10 AM,0,2,0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1149,134.221.96.0-0.0.0.99-0-0-0,134.221.96.0,0,0.0.0.99,0,0,05/05/2026 11:12:07 AM,0,2,0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1177,134.221.96.0-0.0.6.25-0-0-0,134.221.96.0,0,0.0.6.25,0,0,05/05/2026 10:42:18 AM,0,2,0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1274,134.221.96.0-0.0.10.116-0-0-0,134.221.96.0,0,0.0.10.116,0,0,05/05/2026 10:45:00 AM,0,2,0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1709,103.43.91.68-10.0.2.15-443-38366-6,103.43.91.68,443,10.0.2.15,38366,6,05/05/2026 11:51:24 AM,0,2,0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1752,43.249.38.110-10.0.2.15-443-40772-6,43.249.38.110,443,10.0.2.15,40772,6,05/05/2026 11:51:46 AM,0,2,0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1803,74.214.196.131-10.0.2.15-443-58518-6,74.214.196.131,443,10.0.2.15,58518,6,05/05/2026 11:52:09 AM,0,2,0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1808,54.85.61.248-10.0.2.15-443-39686-6,54.85.61.248,443,10.0.2.15,39686,6,05/05/2026 11:52:10 AM,0,2,0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


In [7]:
print("Before:", df.shape)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print("Cleaned dataset shape:", df.shape)

Before: (36210, 84)
Cleaned dataset shape: (36187, 84)


In [8]:
df.Label.value_counts()

Label
Attack    20010
Benign    16177
Name: count, dtype: int64

Need to add more benign ICMP traffic 

For now let's do 15k attack + 13.5k benign in train set and remaining in test

In [9]:
df.describe()

,Src Port,Dst Port,Protocol,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,...,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,36187.000000,36187.000000,36187.000000,3.618700e+04,36187.000000,36187.000000,3.618700e+04,3.618700e+04,36187.000000,36187.000000,...,36187.000000,36187.000000,3.618700e+04,3.618700e+04,3.618700e+04,3.618700e+04,3.618700e+04,3.618700e+04,3.618700e+04,3.618700e+04
mean,12705.379777,184.436566,4.373338,6.389740e+07,19.526128,15.544837,4.638656e+03,1.898923e+04,137.060243,19.081024,...,2.949181,3.343521,3.382024e+06,2.078787e+06,6.154653e+06,1.582158e+06,8.297655e+06,2.911900e+06,1.309164e+07,5.815417e+06
std,21271.817230,2373.536183,7.102876,5.315088e+07,104.465673,326.409973,2.377124e+05,4.717148e+05,881.109486,71.084824,...,49.966785,5.932180,4.127099e+06,3.019632e+06,7.559597e+06,2.607879e+06,1.181323e+07,5.145997e+06,1.516578e+07,1.097996e+07
min,0.000000,0.000000,0.000000,1.000000e+00,1.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000,0.000000,0.000000,2.775249e+06,3.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000,0.000000,0.000000,9.011223e+07,14.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,3.017766e+06,4.857596e+05,3.930651e+06,1.800285e+06,8.163645e+06,2.025587e+06,1.310859e+07,5.554617e+06
75%,35210.500000,53.000000,6.000000,1.163818e+08,21.000000,1.000000,3.200000e+01,3.900000e+01,31.000000,0.000000,...,0.000000,8.000000,5.758353e+06,3.454221e+06,1.067825e+07,1.975106e+06,1.018740e+07,4.220725e+06,1.900373e+07,5.751146e+06
max,65261.000000,60310.000000,17.000000,1.200000e+08,4494.000000,17822.000000,2.549826e+07,4.905276e+07,65160.000000,5840.000000,...,3836.000000,40.000000,1.078944e+08,5.854058e+07,1.078944e+08,1.078944e+08,1.196240e+08,7.657851e+07,1.196240e+08,1.196240e+08


In [11]:
zero_features = ['Bwd PSH Flags',"Fwd URG Flags","Bwd URG Flags","Bwd PSH Flags","Fwd URG Flags",
                 "Bwd URG Flags","URG Flag Count","CWR Flag Count","ECE Flag Count","Fwd Bytes/Bulk Avg","Fwd Packet/Bulk Avg",
                 "Fwd Bulk Rate Avg"]


df_cleaned = df.drop(columns=zero_features)

print(f"\nOriginal shape: {df.shape}")
print(f"Cleaned shape: {df_cleaned.shape}")


Original shape: (36187, 84)
Cleaned shape: (36187, 75)


In [18]:
df_cleaned.columns

Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s',
       'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max',
       'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance',
       'FIN Flag Count', 'SYN Flag Count', 

In [20]:
additional_drop = ['Flow ID','Src IP','Src Port', 'Dst IP','Protocol',
       'Timestamp']

df_cleaned = df_cleaned.drop(columns = additional_drop) 

## Custom Split

In [21]:
attack_df = df_cleaned[df_cleaned["Label"] == "Attack"]
benign_df = df_cleaned[df_cleaned["Label"] == "Benign"]

attack_df = attack_df.sample(frac=1, random_state=42)
benign_df = benign_df.sample(frac=1, random_state=42)

train_attack = attack_df.iloc[:15000]
train_benign = benign_df.iloc[:13500]

train_df = pd.concat([train_attack, train_benign])

test_attack = attack_df.iloc[15000:]
test_benign = benign_df.iloc[13500:]

test_df = pd.concat([test_attack, test_benign])


train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Train set:")
print(train_df["Label"].value_counts())

print("\nTest set:")
print(test_df["Label"].value_counts())

Train set:
Label
Attack    15000
Benign    13500
Name: count, dtype: int64

Test set:
Label
Attack    5010
Benign    2677
Name: count, dtype: int64


In [22]:
X_train = train_df.drop("Label", axis=1)
y_train = train_df["Label"]

X_test = test_df.drop("Label", axis=1)
y_test = test_df["Label"]


In [23]:
le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

print(le.classes_)

['Attack' 'Benign']


## Training Xgboost

In [24]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Optional[float]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[str], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = loa

## Evaluation

In [25]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9970079354754781

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5010
           1       1.00      1.00      1.00      2677

    accuracy                           1.00      7687
   macro avg       1.00      1.00      1.00      7687
weighted avg       1.00      1.00      1.00      7687


Confusion Matrix:
[[5000   10]
 [  13 2664]]


## Conclusion


While although the dataset generated is decent. Few issues I noticed: -

+ 9 zero-variance columns, CICFlowMeter is built primarily for TCP/UDP traffic. By forcing ICMP through a TCP-centric extractor, you get a lot of noise and miss ICMP-specific nuances. For example, it doesn't differentiate between ICMP Type 8 (Echo Request) and Type 0 (Echo Reply) sequences, nor does it track ICMP Code anomalies, which are classic indicators of custom tunneling tools.

+ The Missing Entropy column - BIG miss

+ Need some more ICMP benign capture

Improvement in these areas: -


- Add an entropy feature
- Add type for ICMP packets.WHY? For example, it doesn't differentiate between ICMP Type 8 (Echo Request) and Type 0 (Echo Reply) sequences, nor does it track ICMP Code anomalies, which are classic indicators of custom tunneling tools.
- Track Request/Reply Ratios: Normal ICMP behavior is a 1:1 request-to-reply ratio. Tunnels often skew this (e.g., sending data out via requests but dropping the replies to stay quiet). A simple Total Fwd Packets / Total Bwd Packets custom feature would catch this instantly.
